In [ ]:
import pathlib
import os
import pandas as pd
import numpy as np

In [ ]:
!pip install biopython


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 9.0 MB/s eta 0:00:00


In [ ]:
# Set up file system
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
# download the git repo
!git clone https://github.com/ConorMesser/stats_305c.git
!cd stats_305c/; git pull

Cloning into 'stats_305c'...
remote: Enumerating objects: 59, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 59 (delta 24), reused 45 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (59/59), 14.05 MiB | 8.81 MiB/s, done.
Resolving deltas: 100% (24/24), done.
Already up to date.


# Load Data

In [ ]:
# file_path = '/content/drive/MyDrive/STATS305c/data' # Lauren
file_path = '/content/drive/MyDrive/Courses/STATS305c/data' # Conor

mic_file = pathlib.Path(os.path.join(file_path, 'MIC_data_cleaned.csv'))

full_mic_data = pd.read_csv(mic_file)


# Functions

In [ ]:
taxonomy_input_df = full_mic_data[['Target Species']].copy()
taxonomy_input_df.rename(columns={'Target Species': 'strain'}, inplace=True)
taxonomy_input_df.loc[:, 'species_input'] = taxonomy_input_df.loc[:, 'strain'].str.split().str[:2].str.join(' ')

In [ ]:
from Bio import Entrez
# NCBI requires an email address to use their public API
Entrez.email = "csmesser@stanford.edu"
Entrez.api_key = ""  # FILL IN

import time

def get_taxonomy_batched(species_list):
    tax_ids = []
    tax_ids_dict = {}
    failed_species = []

    print("Step 1: Fetching TaxIDs...")
    for species in species_list:
        try:
            handle = Entrez.esearch(db="taxonomy", term=species)
            record = Entrez.read(handle)
            handle.close()

            if record["IdList"]:
                tax_ids.append(record["IdList"][0])
                tax_ids_dict[record["IdList"][0]] = species
            else:
                failed_species.append(species)
        except Exception as e:
            failed_species.append(species)

        time.sleep(0.11) # Respect NCBI API limits

    # SAFETY CHECK: If no IDs were found, exit early to prevent a 400 error
    if not tax_ids:
        print("No valid TaxIDs found. Exiting.")
        return pd.DataFrame(), failed_species

    print(f"Step 2: Fetching lineages for {len(tax_ids)} species in chunks...")

    records = []
    chunk_size = 100 # NCBI handles chunks of 100 perfectly

    print(tax_ids)

    # CHUNKING LOGIC: Process the IDs in batches of 100
    for i in range(0, len(tax_ids), chunk_size):
        chunk = tax_ids[i:i + chunk_size]
        id_string = ",".join(chunk)

        try:
            fetch_handle = Entrez.efetch(db="taxonomy", id=id_string, retmode="xml")
            chunk_records = Entrez.read(fetch_handle)
            fetch_handle.close()

            # Entrez.read returns a list of dictionaries; add them to our master list
            records.extend(chunk_records)

        except Exception as e:
            print(f"Failed to fetch chunk {i} to {i+chunk_size}: {e}")

        time.sleep(0.11) # Sleep between chunks

    print("Step 3: Parsing data...")
    target_ranks = ['superkingdom', 'domain', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species']
    results = []

    for tax_data in records:
        row_data = {rank: pd.NA for rank in target_ranks}
        current_name = tax_data.get("ScientificName", "")
        this_tax_id = tax_data.get("TaxId", "")
        row_data['input'] = tax_ids_dict[this_tax_id]

        ancestors = tax_data.get("LineageEx", [])
        all_nodes = ancestors + [{"Rank": tax_data.get("Rank", ""), "ScientificName": current_name}]

        for node in all_nodes:
            rank = node.get("Rank", "").lower()
            name = node.get("ScientificName", "")
            if rank in target_ranks:
                row_data[rank] = name

        results.append(row_data)

    df = pd.DataFrame(results)

    return df, failed_species


In [ ]:

# ==========================================
# Example Usage
# ==========================================
# A mix of bacteria, fungi, viruses, and generic IDs
print("Querying NCBI...\n")

# Generate the dataframe
taxonomy_df, failed_species = get_taxonomy_batched(taxonomy_input_df['species_input'].unique())


Querying NCBI...

Step 1: Fetching TaxIDs...
Step 2: Fetching lineages for 732 species in chunks...
['1280', '1283', '29385', '1292', '1423', '562', '28901', '582', '1352', '5476', '4932', '573', '550', '287', '584', '1282', '1351', '36911', '109871', '714', '470', '40324', '546', '5480', '5482', '4909', '1309', '1579', '53413', '585', '5478', '1824', '1747', '571', '646', '1311', '46126', '1933880', '1334', '1349', '146827', '1336', '1396', '1288', '644', '294', '1377', '1270', '669', '55601', '303', '1799160', '1428', '1639', '90371', '1773', '1502', '552', '317', '339', '48664', '56448', '210', '520', '5580', '5059', '746128', '5061', '5507', '5530', '40559', '36656', '100870', '5465', '76777', '727', '615', '5207', '5141', '5516', '984957', '1404', '40215', '633', '29388', '1314', '4896', '55194', '5553', '1305', '1310', '1302', '1656', '1655', '1582', '1613', '29908', '47879', '5599', '169388', '29918', '28447', '554', '56460', '216816', '1304', '1307', '1313', '137621', '666', '6

In [ ]:
taxonomy_df.head()

,superkingdom,domain,kingdom,phylum,class,order,family,genus,species,input
0,<NA>,Bacteria,Bacillati,Bacillota,Bacilli,Caryophanales,Staphylococcaceae,Staphylococcus,Staphylococcus aureus,Staphylococcus aureus
1,<NA>,Bacteria,Bacillati,Bacillota,Bacilli,Caryophanales,Staphylococcaceae,Staphylococcus,Staphylococcus haemolyticus,Staphylococcus haemolyticus
2,<NA>,Bacteria,Bacillati,Bacillota,Bacilli,Caryophanales,Staphylococcaceae,Staphylococcus,Staphylococcus saprophyticus,Staphylococcus saprophyticus
3,<NA>,Bacteria,Bacillati,Bacillota,Bacilli,Caryophanales,Staphylococcaceae,Staphylococcus,Staphylococcus warneri,Staphylococcus warneri
4,<NA>,Bacteria,Bacillati,Bacillota,Bacilli,Caryophanales,Bacillaceae,Bacillus,Bacillus subtilis,Bacillus subtilis


In [ ]:
taxonomy_df.shape

(732, 10)

In [ ]:
len(failed_species)

48

In [ ]:
sorted(failed_species)

['Agrobacterium rhizogenes',
 'Bacillus circulans',
 'Bacteroides vulgatus',
 'Candida guilliermondii',
 'Clavibacter fangii',
 'Clostridium oroticum',
 'Cryptococcus albidus',
 'Cryptococcus cuniculi',
 'Enterobacter spp.',
 'Eubacterium rectale',
 'Gibberella saubinetii',
 'Haemophilus spp.',
 'Human acute',
 'Human breast',
 'Human cervical',
 'Human gastric',
 'Human lung',
 'Human myelogenous',
 'Human ovarian',
 'Human pancreatic',
 'Human prostate',
 'Human skin',
 'Lactococcus raffinolactis',
 'Moraxella spp.',
 'Mycobacterium smegmatis',
 'Mycoplasma hominis',
 'Neisseria spp.',
 'Nocardia spp.',
 'Paecilomyces spp.',
 'Parageobacillus toebi',
 'Peptostreptococcus micros',
 'Prevotella copri',
 'Pseudomonas oleovorans',
 'Pseudomonas stutzeri',
 'Rhodococcus equi',
 'Rhodococcus fascians',
 'Rhodococcus sp.',
 'Salmonella Bazenheid',
 'Salmonella bonariensis',
 'Serratia sp.',
 'Slime mold',
 'Staphylococcus citreus',
 'Staphylococcus sciuri',
 'Streptococcus Sc181',
 'Strepto

In [ ]:
taxonomy_output_df = taxonomy_input_df.merge(taxonomy_df, how='left', left_on='species_input', right_on='input')


In [ ]:
sorted(taxonomy_output_df[taxonomy_output_df['genus'].isnull()]['species_input'].unique())

['Acinetobacter junii',
 'Aeromonas caviae',
 'Agrobacterium rhizogenes',
 'Bacillus circulans',
 'Bacteroides vulgatus',
 'Candida guilliermondii',
 'Candida kefyr',
 'Candida sp.',
 'Clavibacter fangii',
 'Clostridium oroticum',
 'Cryptococcus albidus',
 'Cryptococcus cuniculi',
 'Enterobacter spp.',
 'Eubacterium rectale',
 'Gibberella saubinetii',
 'Haemophilus influenzae',
 'Haemophilus spp.',
 'Human acute',
 'Human breast',
 'Human cervical',
 'Human gastric',
 'Human lung',
 'Human myelogenous',
 'Human ovarian',
 'Human pancreatic',
 'Human prostate',
 'Human skin',
 'Hypocreales sp.',
 'Lactococcus raffinolactis',
 'Moraxella spp.',
 'Mycobacterium smegmatis',
 'Mycoplasma hominis',
 'Neisseria spp.',
 'Nocardia spp.',
 'Paecilomyces spp.',
 'Parageobacillus toebi',
 'Peptostreptococcus micros',
 'Phytophthora nicotianae',
 'Prevotella copri',
 'Pseudomonas oleovorans',
 'Pseudomonas stutzeri',
 'Rhodococcus equi',
 'Rhodococcus fascians',
 'Rhodococcus sp.',
 'Salmonella Baz

In [ ]:
{val: '' for val in taxonomy_output_df[taxonomy_output_df['genus'].isna()]['species_input'].value_counts().sort_values().iloc[-18:][::-1].index}

{'Mycobacterium smegmatis': '',
 'Bacteroides vulgatus': '',
 'Candida kefyr': '',
 'Streptococcus group': '',
 'Haemophilus influenzae': '',
 'Eubacterium rectale': '',
 'Prevotella copri': '',
 'Staphylococcus sciuri': '',
 'Streptococcus Sc181': '',
 'Pseudomonas stutzeri': '',
 'Slime mold': '',
 'Candida guilliermondii': '',
 'Acinetobacter junii': '',
 'Serratia sp.': '',
 'Rhodococcus equi': '',
 'Aeromonas caviae': '',
 'Peptostreptococcus micros': '',
 'hotobacterium damselae': ''}

In [ ]:
na_species = {  # From Gemini
    # --- Recently reclassified ---
    'Mycobacterium smegmatis': 'Mycolicibacterium smegmatis',
    'Bacteroides vulgatus': 'Phocaeicola vulgatus',
    'Candida kefyr': 'Kluyveromyces marxianus',
    'Eubacterium rectale': 'Agathobacter rectalis',
    'Prevotella copri': 'Segatella copri',
    'Staphylococcus sciuri': 'Mammaliicoccus sciuri',
    'Pseudomonas stutzeri': 'Stutzerimonas stutzeri',
    'Candida guilliermondii': 'Meyerozyma guilliermondii',
    'Rhodococcus equi': 'Prescottia equi',
    'Peptostreptococcus micros': 'Parvimonas micra',

    # --- Formatting, Typos, and Unresolved terms ---
    'Streptococcus group': 'Streptococcus',
    'Streptococcus Sc181': 'Streptococcus',
    'Slime mold': 'Mycetozoa',
    'Serratia sp.': 'Serratia',
    'hotobacterium damselae': 'Photobacterium damselae',

    # --- Capitalization fixed (Already valid in NCBI) ---
    'Haemophilus influenzae': 'Haemophilus influenzae',
    'Acinetobacter junii': 'Acinetobacter junii',
    'Aeromonas caviae': 'Aeromonas caviae'

}

taxonomy_failed_df, failed_x2_species = get_taxonomy_batched(na_species.values())


Step 1: Fetching TaxIDs...
Step 2: Fetching lineages for 17 species in chunks...
['1772', '821', '4911', '39491', '165179', '1296', '316', '4929', '33033', '1301', '1301', '142796', '2985502', '38293', '727', '40215', '648']
Step 3: Parsing data...


In [ ]:
tmp = taxonomy_failed_df.copy()
tmp['input'] = tmp['input'].map({v: k for k, v in na_species.items()})
tax_df_w_failed = pd.concat([taxonomy_df, tmp],axis=0).drop_duplicates()

# Add Streptococcus group values
tax_df_w_failed.loc[len(tax_df_w_failed)] = [np.nan, 'Bacteria', 'Bacillati', 'Bacillota', 'Bacilli',
       'Lactobacillales', 'Streptococcaceae', 'Streptococcus',
       np.nan, 'Streptococcus group']

taxonomy_output_df = taxonomy_input_df.merge(tax_df_w_failed, how='left', left_on='species_input', right_on='input')
taxonomy_output_df = taxonomy_output_df[['superkingdom','domain','kingdom','phylum','class','order','family','genus','species','strain']].bfill(axis=1)


In [ ]:
taxonomy_output_df.to_csv(os.path.join(file_path, 'species_taxonomy.tsv'), sep='\t', index=False)